(py-conversion)=
# Vectorize und Rasterize

This section introduces geospatial dataset conversion with Python. In particular, the goal of this section is to guide to an understanding of conversions from raster to vector data formats and vice versa. For interactive reading and executing code blocks [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/hydro-informatics/jupyter-python-course/main?filepath=jupyter) and find *geo03-conversion.ipynb*, or install {ref}`Python <install-python>` and {ref}`JupyterLab <jupyter>` locally.

```{admonition} Requirements
* Stellen Sie sicher, {ref}`handling of gridded raster data <py-raster>` und {ref}`shapefile handling <chpt-shp>`
* Die Erstellung des {ref}`least cost path <leastcost>` raster datasets verstehen.
```

```{admonition} Tips
:class: tip
* Die in diesem Abschnitt enthaltenen Funktionen werden teilweise auch in [flusstools](https://flusstools.readthedocs.io).
* Um diese Funktionen zu nutzen, stellen Sie sicher, dass flusstools installiert ist und wie folgt importiert wird: `from flusstools import geotools`. Einige der in diesem Tutorial gezeigten Funktionen können dann mit `geotools.function_name()` verwendet werden.
```

```{admonition} Watch this section as a video
:class: tip, dropdown
<iframe width="701" height="394" src="https://www.youtube-nocookie.com/embed/gsurU_DDhBc" title="YouTube video player" frameborder="0" allow="accelerometer; autoplay; encrypted-media; gyroscope; picture-in-picture" allowfullscreen></iframe>
<p>Watch this section as a video on the <a href="https://www.youtube.com/@hydroinformatics">@Hydro-Morphodynamics channel on YouTube</a>.</p>
```

## Import relevante Bibliotheken

Stellen Sie sicher, die relevanten Pakete für die Verarbeitung von Rastern, Formdateien und geospatialen Referenzen zu importieren:

In [ ]:
from osgeo import gdal
from osgeo import osr
from osgeo import ogr

## Vectoris

(raster2line)=
### Raster nach Linie

In diesem Abschnitt konvertieren wir den {ref}`least cost path <leastcost>` raster dataset ([least cost.tif](https://github.com/hydro-informatics/jupyter-python-course/raw/main/geodata/rasters/least-cost.tif)) in eine (Poly)-Line Shapefile. Dazu schreiben wir zunächst eine Funktion namens `offset2coords()`, die das Inverse der {ref}`coords2offset() <lc-fun>`-Funktion darstellt, und wandelt x/y Offset (in *integer* Pixelnummern) in Koordinaten der Geotransformation eines Geodatensatzes um:

In [1]:
def offset2coords(geo_transform, offset_x, offset_y):
    # get origin and pixel dimensions from geo_transform (osgeo.gdal.Dataset.GetGeoTransform() object)
    origin_x = geo_transform[0]
    origin_y = geo_transform[3]
    pixel_width = geo_transform[1]
    pixel_height = geo_transform[5]
    
    # calculate x and y coordinates
    coord_x = origin_x + pixel_width * (offset_x + 0.5)
    coord_y = origin_y + pixel_height * (offset_y + 0.5)

    # return x and y coordinates
    return coord_x, coord_y

```{note}
Der Offset wird in x- und y-Richtung 0,5 Pixel hinzugefügt, um die Mitte des Pixels anstatt der oberen linken Pixelecke zu erfüllen.
```

Als nächstes können wir eine Kernfunktion schreiben, um einen Rasterdatensatz in eine Zeilenformdatei zu konvertieren. Wir nennen diese Funktion `raster2line()` und sie baut auf dem folgenden Workflow:

* Open a `raster`, its band as `array`, and `geo_transform` (geo-transformation) defined with the `raster_file_name` argument in the {ref}`open_raster <open-raster>` function from the raster section.
* Calculate the maximum distance (`max_distance`) between two pixels that are considered being *connect-able*, based on the hypothesis that the pixel height *&Delta;y* and width *&Delta;x* are the same:
    ![img](../img/pixel2line-width-illu.png)
* Get the `trajectory` of pixels that have a user parameter-defined `pixel_value` (e.g., `1` to trace 1-pixels in the binary *least_cost.tif*) and throw an error if the trajectory is empty (i.e., `np.count_nonzero(trajectory) == 0`). 
* Use the above-defined `offset2coords` function to append point coordinates to a `points` list.
* Create a `multi_line` object (instance of `ogr.Geometry(ogr.wkbMultiLineString)`), which represents the (void) final least cost path.
* Iterate through all possible combinations of points (excluding combinations of points with themselves) with [`itertools.combinations(iterable, r=number-of-combinations=2`](https://docs.python.org/3/library/itertools.html)).

- Punkte werden in der Liste `points` gespeichert.
- `point1` und `point2` sind erforderlich, um den Abstand zwischen Punktenpaaren zu erhalten.
- Wenn die `distance` zwischen den Punkten kleiner ist als `max_distance`, erstellt die Funktion ein Zeilenobjekt aus den beiden Punkten und legt es an das `multi_line`Objekt an.

* Erstellen Sie eine neue Formdatei (Name `out_shp_fn`) unter Verwendung der {ref}`create_shp() <create-shp>`-Funktion (mit integrierter Formfile Name Längenverifikation von flusstools `geotools.create_shp()`).
* Fügen Sie das `multi_line`-Objekt als neue Funktion in die Formdatei ein (gemäß den Beschreibungen im {ref}`shapefile section <line-create>`).
* Erstellen Sie eine `.prj`-Projektionsdatei (Recall-Beschreibungen in der {ref}`shapefile section <prj-shp>`) unter Verwendung des räumlichen Referenzsystems der Eingabe `raster` mit der {ref}`get_srs() <lc-fun>`-Funktion.

The `raster2line` function is also implemented in the [`flusstools.geotools.geotools`](https://raw.githubusercontent.com/Ecohydraulics/flusstools-pckg/main/flusstools/geotools/geotools.py) script.

In [2]:
import os
import itertools
import numpy as np
from osgeo import ogr
from flusstools.geotools import (
    raster2array, open_raster, create_raster, create_shp, get_srs, make_prj,
)


def raster2line(raster_file_name, out_shp_fn, pixel_value):
    """
    Convert a raster to a line shapefile, where pixel_value determines line start and end points
    :param raster_file_name: STR of input raster file name, including directory; must end on ".tif"
    :param out_shp_fn: STR of target shapefile name, including directory; must end on ".shp"
    :param pixel_value: INT/FLOAT of a pixel value
    :return: None (writes new shapefile).
    """

    # calculate max. distance between points
    # ensures correct neighbourhoods for start and end pts of lines
    raster, array, geo_transform = raster2array(raster_file_name)
    pixel_width = geo_transform[1]
    max_distance = np.ceil(np.sqrt(2 * pixel_width**2))

    # extract pixels with the user-defined pixel value from the raster array
    trajectory = np.where(array == pixel_value)
    if np.count_nonzero(trajectory) == 0:
        print("ERROR: The defined pixel_value (%s) does not occur in the raster band." % str(pixel_value))
        return None

    # convert pixel offset to coordinates and append to nested list of points
    points = []
    count = 0
    for offset_y in trajectory[0]:
        offset_x = trajectory[1][count]
        points.append(offset2coords(geo_transform, offset_x, offset_y))
        count += 1

    # create multiline (write points dictionary to line geometry (wkbMultiLineString)
    multi_line = ogr.Geometry(ogr.wkbMultiLineString)
    for i in itertools.combinations(points, 2):
        point1 = ogr.Geometry(ogr.wkbPoint)
        point1.AddPoint(i[0][0], i[0][1])
        point2 = ogr.Geometry(ogr.wkbPoint)
        point2.AddPoint(i[1][0], i[1][1])

        distance = point1.Distance(point2)
        if distance < max_distance:
            line = ogr.Geometry(ogr.wkbLineString)
            line.AddPoint(i[0][0], i[0][1])
            line.AddPoint(i[1][0], i[1][1])
            multi_line.AddGeometry(line)

    # write multiline (wkbMultiLineString2shp) to shapefile
    new_shp = create_shp(out_shp_fn, layer_name="raster_pts", layer_type="line")
    lyr = new_shp.GetLayer()
    feature_def = lyr.GetLayerDefn()
    new_line_feat = ogr.Feature(feature_def)
    new_line_feat.SetGeometry(multi_line)
    lyr.CreateFeature(new_line_feat)

    # create projection file
    srs = get_srs(raster)
    make_prj(out_shp_fn, int(srs.GetAuthorityCode(None)))
    print("Success: Wrote %s" % str(out_shp_fn))

Die `raster2line()`-Funktion kann wie folgt aufgerufen werden, um den kostengünstigsten Pfad vom Pixel (raster) in das Zeilenformat (Vektor) umzuwandeln:

In [3]:
source_raster_fn = r"" +  os.path.abspath("") + "/geodata/rasters/least-cost.tif"
target_shp_fn = r"" + os.path.abspath("") + "/geodata/shapefiles/least-cost.shp"
pixel_value = 1
raster2line(source_raster_fn, target_shp_fn, pixel_value)

Success: Wrote /home/schwindt/jupyter/geodata/shapefiles/least_cost.shp


```{figure} ../img/qgis-least-cost-line.png
:alt: python convert raster to line
:name: qgis-least-cost-line-py

Der Raster des am wenigsten kostenpfads, der in eine Linienformdatei umgewandelt wird, zeigt in QGIS.
```

```{admonition} Challenge
Es gibt einen kleinen Fehler in der `least_cost`-Linie. Finden Sie den Fehler? Was kann getan werden, um den Fehler zu beheben?
```

```{note}
Network Routing ist der Funktionskern der {ref}`NetworkX library (see *Open source libraries*) <other-geo-pckgs>`. Lesen Sie mehr über Netzwerkanalysen auf [Michael Diener's GitHub page](https://github.com/mdiener21/python-geospatial-analysis-cookbook/tree/master/ch08).
```

(raster2polygon)=
### Raster nach Polygon

`gdal` kommt mit der leistungsstarken `Polygonize`-Funktion für die einfache Umwandlung eines Rasterdatensatzes in eine Polygonformdatei. `gdal.Polygonize` ermöglicht es, einfach zu schreiben `raster2polygon()` Python-Funktion, es hat den Nachteil, dass es nur ganzzahlige Werte verarbeiten kann und es nur zufällig `FID` (automatisch zugeschriebene Kennungen ohne physikalische Bedeutung) Werte standardmäßig Attribute. Da die `FID`-Werte nicht aussagekräftig sind, erstellen wir eine `float2int()`-Funktion, um den ursprünglichen Wertebereich zu erhalten (verwendet die Funktionen {ref}`raster2array() <createarray>` und {ref}`create_raster() <create-raster>` aus dem Rasterbereich):

In [4]:
def float2int(raster_file_name, band_number=1):
    """
    :param raster_file_name: STR of target file name, including directory; must end on ".tif"
    :param band_number: INT of the raster band number to open (default: 1)
    :output: new_raster_file_name (STR)
    """
    # use raster2array function to get raster, np.array and the geo transformation
    raster, array, geo_transform = raster2array(raster_file_name, band_number=band_number)
    
    # convert np.array to integers
    try:
        array = array.astype(int)
    except ValueError:
        print("ERROR: Invalid raster pixel values.")
        return raster_file_name
    
    # get spatial reference system
    src_srs = get_srs(raster)
    
    # create integer raster    
    new_name = raster_file_name.split(".tif")[0] + "_int.tif"
    create_raster(new_name, array, epsg=int(src_srs.GetAuthorityCode(None)),
                  rdtype=gdal.GDT_Int32, geo_info=geo_transform)
    # return name of integer raster
    return new_name

Als nächstes erstellen wir die `raster2polygon()`-Funktion, die den folgenden Workflow implementiert:

1. Use the `float2int()` function to ensure that any raster `file_name` provided can be converted to purely integer values.
1. Create a new shapefile (named `out_shp_fn`) using the {ref}`create_shp() <create-shp>` function (also available from flusstools: `geotools.create_shp()`).
1. Add a new `ogr.OFTInteger` field (recall {ref}`how field creation works <add-field>`) in the shapefile section) named by the optional `field_name` input argument.
1. Run [`gdal.Polygonize`](https://gdal.org/api/gdal_alg.html#_CPPv414GDALPolygonize15GDALRasterBandH15GDALRasterBandH9OGRLayerHiPPc16GDALProgressFuncPv) with:

    * `hSrcBand=raster_band`
    * `hMaskBand=None` (optional raster band to define polygons)
    * `hOutLayer=dst_layer`
    * `iPixValField=0` (if no field was added, set to `-1` in order to create an `FID` field; if more fields were added, set to `1`, `2`, ... )
    * `papszOptions=[]` (no effect for `ESRI Shapefile` driver type)
    * `callback=None` for not using the reporting algorithm (`GDALProgressFunc()`)

1. Erstellen Sie eine `.prj`-Projektionsdatei (Recall-Beschreibungen in der {ref}`shapefile section <prj-shp>`) unter Verwendung des räumlichen Referenzsystems der Eingabe `raster` mit der {ref}`get_srs() <lc-fun>`-Funktion.

In [5]:
def raster2polygon(file_name, out_shp_fn, band_number=1, field_name="values"):
    """
    Convert a raster to polygon
    :param file_name: STR of target file name, including directory; must end on ".tif"
    :param out_shp_fn: STR of a shapefile name (with directory e.g., "C:/temp/poly.shp")
    :param band_number: INT of the raster band number to open (default: 1)
    :param field_name: STR of the field where raster pixel values will be stored (default: "values")
    :return: None
    """
    # ensure that the input raster contains integer values only and open the input raster
    file_name = float2int(file_name)
    raster, raster_band = open_raster(file_name, band_number=band_number)

    # create new shapefile with the create_shp function
    new_shp = create_shp(out_shp_fn, layer_name="raster_data", layer_type="polygon")
    dst_layer = new_shp.GetLayer()

    # create new field to define values
    new_field = ogr.FieldDefn(field_name, ogr.OFTInteger)
    dst_layer.CreateField(new_field)

    # Polygonize(band, hMaskBand[optional]=None, destination lyr, field ID, papszOptions=[], callback=None)
    gdal.Polygonize(raster_band, None, dst_layer, 0, [], callback=None)

    # create projection file
    srs = get_srs(raster)
    make_prj(out_shp_fn, int(srs.GetAuthorityCode(None)))
    print("Success: Wrote %s" % str(out_shp_fn))

```{admonition} Note
:class: tip
* `Polygonize` can also be run from {ref}`terminal/Anaconda prompt <terminal>` by typing [`gdal_polygonize`](https://gdal.org/programs/gdal_polygonize.html).
* Both the `float2int()` and the `raster2polygon()` functions are also available in flusstools with `flusstools.geotools.float2int()` and `flusstools.geotools.raster2polygon()` respectively ([have a look at the geotools.py script](https://raw.githubusercontent.com/Ecohydraulics/flusstools-pckg/main/flusstools/geotools/geotools.py)).
```

Die `raster2polygon()`-Funktion kann z.B. implementiert werden, um den Wassertiefenraster für 1000 CFS (*h001000.tif* vom [*River Architect* Sample datasets](https://github.com/RiverArchitect/SampleData/tree/master/01_Conditions/2100_sample)) in eine Polygon-Formdatei umzuwandeln:

In [6]:
src_raster = r"" +  os.path.abspath("") + "/geodata/rasters/h001000.tif"
tar_shp = r"" + os.path.abspath("") + "/geodata/shapefiles/h_poly_cls.shp"
raster2polygon(src_raster, tar_shp)

Success: Wrote /home/schwindt/jupyter/geodata/shapefiles/h_poly_cls.shp


```{figure} ../img/qgis-h-polygonized.png
:alt: python convert raster to polygon shapefile
:name: qgis-h-polygonized-py

Das Raster der Wassertiefen umgewandelt in eine Polygon-Formdatei mit Zonen, in QGIS gezeigt.
```

(shp2raster)=
## Rasterize (Vector Shapefile zu Raster)

Similar to `gdal.Polygonize`, [`gdal.RasterizeLayer`](https://gdal.org/python/osgeo.gdal-module.html#RasterizeLayer) represents a handy option to convert a shapefile into a raster. However, to be precise, a shapefile is not really converted into a raster but burned onto a raster. Thus, values stored in a field of a shapefile feature are used (burned) as pixel values for creating a new raster. Attention is required to ensure that the correct values and data types are used. To this end, the below shown `rasterize()` function implements the following workflow that avoids potential conversion headaches:

1. Öffnen Sie den Benutzer bereitgestellten Eingabe-Formdateinamen und -schicht.
1. Lesen Sie die räumliche Ausdehnung der Schicht.
1. Ableiten der x-y-Auflösung in Abhängigkeit von der räumlichen Ausdehnung und einer benutzerdefinierten `pixel_size` (optionales Keyword Argument mit Standardwert).
1. Erstellen Sie einen neuen GeoTIFF-Raster mit
* benutzerdefinierte `output_raster_file_name`,
* berechnete x- und y-Auflösung und
* `eType` (default ist `gdal.GDT_Float32` - Rufen Sie an alle Datentyp-Optionen, die in der {ref}`raster section <etypes>` aufgeführt sind.
1. Tragen Sie die durch die Quellschichtverläufe definierte Geotransformation und die `pixel_size` an.
1. Erstellen Sie einen raster `band`, füllen Sie die `band` mit dem benutzerdefinierten `no_data_value` (Standard ist `-9999`) und setzen Sie die `no_data_value`.
1. Legen Sie das räumliche Referenzsystem des Rasters auf das gleiche wie die Quellformdatei.
1. Apply `gdal.RasterizeLayer`
* `dataset=target_ds` (Ziel-Raster-Datensatz)
* `bands=[1]` (*list(integer)* - erhöhen Sie auf definierte mehr Rasterbänder und vergeben andere Werte, zum Beispiel aus anderen Feldern der Quellformdatei),
* `layer=source_lyr` (Layer mit Features zum Verbrennen an den Raster),
*`pfnTransformer=None` ([weiterlesen in den gdal docs](https://gdal.org/api/python/osgeo.gdal.html?highlight=rasterize#osgeo.gdal.Rasterize))
*`pTransformArg=None` ([weiterlesen in den gdal docs](https://gdal.org/api/python/osgeo.gdal.html?highlight=rasterize#osgeo.gdal.Rasterize))
* `burn_values=[0]` (ein Standardwert, der an den Raster gebrannt wird),
* `options=["ALL_TOUCHED=TRUE"]` definiert, dass alle Pixel, die von einem Polygon berührt werden, den Feldwert des Polygons erhalten - wenn nicht eingestellt: nur Pixel, die vollständig im Polygon sind, einen Wert zugewiesen werden,
* `options=["ATTRIBUTE=" + str(kwargs.get("field_name"))]` definiert den Feldnamen mit Werten zu verbrennen.

In [18]:
def rasterize(in_shp_file_name, out_raster_file_name, pixel_size=10, no_data_value=-9999,
              rdtype=gdal.GDT_Float32, **kwargs):
    """
    Converts any shapefile to a raster
    :param in_shp_file_name: STR of a shapefile name (with directory e.g., "C:/temp/poly.shp")
    :param out_raster_file_name: STR of target file name, including directory; must end on ".tif"
    :param pixel_size: INT of pixel size (default: 10)
    :param no_data_value: Numeric (INT/FLOAT) for no-data pixels (default: -9999)
    :param rdtype: gdal.GDALDataType raster data type - default=gdal.GDT_Float32 (32 bit floating point)
    :kwarg field_name: name of the shapefile's field with values to burn to the raster
    :return: produces the shapefile defined with in_shp_file_name
    """

    # open data source
    try:
        source_ds = ogr.Open(in_shp_file_name)
    except RuntimeError as e:
        print("Error: Could not open %s." % str(in_shp_file_name))
        return None
    source_lyr = source_ds.GetLayer()

    # read extent
    x_min, x_max, y_min, y_max = source_lyr.GetExtent()

    # get x and y resolution
    x_res = int((x_max - x_min) / pixel_size)
    y_res = int((y_max - y_min) / pixel_size)

    # create destination data source (GeoTIff raster)
    target_ds = gdal.GetDriverByName('GTiff').Create(out_raster_file_name, x_res, y_res, 1, eType=rdtype)
    target_ds.SetGeoTransform((x_min, pixel_size, 0, y_max, 0, -pixel_size))
    band = target_ds.GetRasterBand(1)
    band.Fill(no_data_value)
    band.SetNoDataValue(no_data_value)

    # get spatial reference system and assign to raster
    srs = get_srs(source_ds)
    try:
        srs.ImportFromEPSG(int(srs.GetAuthorityCode(None)))
    except RuntimeError as e:
        print(e)
        return None
    target_ds.SetProjection(srs.ExportToWkt())

    # RasterizeLayer(Dataset dataset, int bands, Layer layer, pfnTransformer=None, pTransformArg=None,
    # int burn_values=0, options=None, GDALProgressFunc callback=0, callback_data=None)
    gdal.RasterizeLayer(target_ds, [1], source_lyr, None, None, burn_values=[0],
                        options=["ALL_TOUCHED=TRUE", "ATTRIBUTE=" + str(kwargs.get("field_name"))])

    # release raster band
    band.FlushCache()

```{tip} 
`Rasterize` can also be run from {ref}`terminal/Anaconda prompt <terminal>` with [`gdal_rasterize`](https://gdal.org/programs/gdal_rasterize.html).
```

Schließlich kann die `rasterize()`-Funktion aufgerufen werden, die polygonisierte Wassertiefe Polygon Shapefile */geodata/shapefiles/h poly cls.shp* ([download es als zip file](https://github.com/hydro-informatics/jupyter-python-course/raw/main/geodata/shapefiles/h_poly_cls.zip)) zurück zu einem Raster (dies ist praktisch nutzlos, aber eine illustrative Übung). Achten Sie auf den Datentyp, der `gdal.GDT_Int32` in Kombination mit dem korrekt definierten `field_name`Argument ist.

In [19]:
src_shp = r"" + os.path.abspath("") + "/geodata/shapefiles/h_poly_cls.shp"
tar_ras = r"" +  os.path.abspath("") + "/geodata/rasters/h_re_rastered.tif"
rasterize(src_shp, tar_ras, pixel_size=5, rdtype=gdal.GDT_Int32, field_name="values")

```{figure} ../img/qgis-h-rasterized.png
:alt: python convert polygon to raster with rasterize
:name: qgis-h-rasterized-py

Der rekonvertierte Raster von Wassertiefen basierend auf der in QGIS gezeigten Polygon-Formdatei mit Tiefenzonen.
```

```{admonition} Exercise
Vertraut mit der Umwandlung von Rastern und Formdateien in der {ref}`geospatial ecohydraulics <ex-geco>` Übung.
```